In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#### Main DF is the data from the World Happiness Report ####

In [2]:
main_df = pd.read_csv('WHR25_Data_Figure_2.1v3.csv', encoding = 'latin-1')
main_df = main_df[['Year', 'Rank', 'Country name', 'Life evaluation (3-year average)', 'Lower whisker', 'Upper whisker']]

In [3]:
main_df.head()

,Year,Rank,Country name,Life evaluation (3-year average),Lower whisker,Upper whisker
0,2024.0,147.0,Afghanistan,1.364,1.301,1.427
1,2023.0,143.0,Afghanistan,1.721,1.667,1.775
2,2022.0,137.0,Afghanistan,1.859,1.795,1.923
3,2021.0,146.0,Afghanistan,2.404,2.339,2.469
4,2020.0,150.0,Afghanistan,2.523,2.449,2.596


#### Stats df comes from the World Bank Group ####

In [4]:
stats = pd.read_csv('updatedData.csv', dtype = str)

In [5]:
stats.head()

,Time,Time Code,Country Name,Country Code,Access to clean fuels and technologies for cooking (% of population) [EG.CFT.ACCS.ZS],"Access to clean fuels and technologies for cooking, rural (% of rural population) [EG.CFT.ACCS.RU.ZS]","Access to clean fuels and technologies for cooking, urban (% of urban population) [EG.CFT.ACCS.UR.ZS]",Access to electricity (% of population) [EG.ELC.ACCS.ZS],"Access to electricity, rural (% of rural population) [EG.ELC.ACCS.RU.ZS]","Access to electricity, urban (% of urban population) [EG.ELC.ACCS.UR.ZS]",...,"Wage and salaried workers, male (% of male employment) (modeled ILO estimate) [SL.EMP.WORK.MA.ZS]","Wage and salaried workers, total (% of total employment) (modeled ILO estimate) [SL.EMP.WORK.ZS]",Women Business and the Law Index Score (scale 1-100) [SG.LAW.INDX],Women who were first married by age 15 (% of women ages 20-24) [SP.M15.2024.FE.ZS],Women who were first married by age 18 (% of women ages 20-24) [SP.M18.2024.FE.ZS],Women's share of population ages 15+ living with HIV (%) [SH.DYN.AIDS.FE.ZS],Young people (ages 15-24) newly infected with HIV [SH.HIV.INCD.YG],"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate) [SL.TLF.CACT.MA.ZS]",Lower secondary school starting age (years) [SE.SEC.AGES],"Prevalence of overweight, weight for height (modeled estimate, % of children under 5) [SH.STA.OWGH.ME.ZS]"
0,2023,YR2023,Afghanistan,AFG,38.6,20.4,87,85.3,81.4,96,...,17.53213228,16.88328333,31.875,9.6,28.7,9.718908163,500,70.074,13,4.4
1,2023,YR2023,Albania,ALB,86.1,72.5,95.2,100,100,100,...,47.61730477,49.0432112,91.25,..,..,27.19631999,100,70.068,11,15.9
2,2023,YR2023,Algeria,DZA,99.7,99.1,99.9,100,100,100,...,68.24910624,69.46448445,57.5,..,..,46.79447954,760,67.61,11,12.5
3,2023,YR2023,Angola,AGO,50.2,8.4,75,51.1,..,77.9,...,33.59939786,24.10708681,79.375,..,..,72.93464637,6600,77.588,12,4.3
4,2023,YR2023,Argentina,ARG,99.9,98.2,99.9,100,100,100,...,72.11395165,74.24670928,79.375,..,..,35.59855781,1100,71.944,12,13.7


#### Standardizing Names Across Data Sets ####

In [6]:
names_fixes = {
    "Congo, Dem. Rep.": "DR Congo",
    "Congo, Rep.": "Congo",
    "Moldova": "Republic of Moldova",
    "Iran, Islamic Rep.": "Iran",
    "Korea, Rep.": "Republic of Korea",
    "Swaziland": "Eswatini",
    "Yemen, Rep.": "Yemen",
    "Puerto Rico (US)": "Puerto Rico",
    "Egypt, Arab Rep.": "Egypt",
    "Turkiye":"Türkiye",
    "Cote d'Ivoire": "Côte dIvoire",
    "Slovak Republic": "Slovakia",
    "Kyrgyz Republic":"Kyrgyzstan",
    "Hong Kong SAR, China":"Hong Kong SAR of China",
    "Venezuela, RB": "Venezuela",
    "Gambia, The": "Gambia",
    "Somalia, Fed. Rep.": "Somalia",
    "Syrian Arab Republic": "Syria"
}

stats["Country Name"] = stats["Country Name"].replace(names_fixes)
stats['Time'] = stats['Time'].astype(int)


#### Joining the WHR Data and WBG Data ####

In [7]:
final_df = main_df.merge(
    stats,
    left_on=["Country name", "Year"],
    right_on=["Country Name", "Time"],
    how="left"
)

#### Dropping Empty Rows and Unnecessary Columns ####

In [8]:
final_df.dropna(subset=["Year"], inplace=True)
final_df = final_df.drop(columns=["Country Name", 'Time', 'Lower whisker', 'Upper whisker', 'Country Code'])

#### DF Wide is data from the UN's Human Development Index ####

In [9]:
df_wide = pd.read_csv('HDI_Data_Pivoted.csv')

#### Renaming the HDI Data to Match with the others ####

In [10]:
country_map = {
    "Bolivia (Plurinational State of)": "Bolivia",
    "Côte d'Ivoire": "Côte dIvoire",
    "Congo (Democratic Republic of the)": "DR Congo",
    "Eswatini (Kingdom of)": "Eswatini",
    "Hong Kong, China (SAR)": "Hong Kong SAR of China",
    "Iran (Islamic Republic of)": "Iran",
    "Lao People's Democratic Republic": "Lao PDR",
    "Korea (Republic of)": "Republic of Korea",
    "Moldova (Republic of)": "Republic of Moldova",
    "Palestine, State of": "State of Palestine",
    "Syrian Arab Republic": "Syria",
    "Tanzania (United Republic of)": "Tanzania",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
}

df_wide["country"] = df_wide["country"].replace(country_map)

#### Creating the Final Merged Dataset, Excluding 2024 As No Data is Present in the WHR ####

In [11]:
final_df = final_df.rename(
    columns={
        "Year": "year",
        "Country name": "country"
    }
)

final_df["year"] = final_df["year"].astype(int)
df_wide["year"] = df_wide["year"].astype(int)


merged_df = final_df.merge(
    df_wide,
    on=["country", "year"],
    how="left"   
)

merged_df["_merge"] = final_df.merge(
    df_wide,
    on = ["country", "year"],
    how="left",
    indicator=True
)["_merge"]

merged_df["_merge"].value_counts()
merged_df = merged_df[merged_df["year"] != 2024]

#### Records that were unable to merge properly ####

In [12]:
merged_df["_merge"].value_counts()

_merge
both          1778
left_only       44
right_only       0
Name: count, dtype: int64

#### Dropping Countries/Territories Not in the WHR ####

In [13]:
countries_to_drop = [
    "Taiwan Province of China",
    "Swaziland",
    "Somaliland Region",
    "North Cyprus",
    "Puerto Rico",
    "Kosovo",
]

merged_df = merged_df[~merged_df["country"].isin(countries_to_drop)]

merged_df['_merge'].value_counts()

_merge
both          1778
left_only        0
right_only       0
Name: count, dtype: int64

In [14]:
merged_df.isnull().sum()

year                                                                                 0
Rank                                                                                 0
country                                                                              0
Life evaluation (3-year average)                                                     0
Time Code                                                                          154
                                                                                  ... 
Population with at least some secondary education, female (% ages 25 and older)     36
Population with at least some secondary education, male (% ages 25 and older)       36
Share of seats in parliament, female (% held by women)                              24
Share of seats in parliament, male (% held by men)                                  24
_merge                                                                               0
Length: 620, dtype: int64

In [15]:
merged_df = merged_df.replace("..", np.nan)
merged_df = merged_df.apply(pd.to_numeric, errors="ignore")
merged_df = merged_df.drop(columns=['Time Code', '_merge'])
merged_df.isnull().sum()

C:\Users\Nick\AppData\Local\Temp\ipykernel_3204\601293837.py:2: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  merged_df = merged_df.apply(pd.to_numeric, errors="ignore")


year                                                                                       0
Rank                                                                                       0
country                                                                                    0
Life evaluation (3-year average)                                                           0
Access to clean fuels and technologies for cooking (% of population) [EG.CFT.ACCS.ZS]    189
                                                                                        ... 
Life Expectancy at Birth, male (years)                                                     0
Population with at least some secondary education, female (% ages 25 and older)           36
Population with at least some secondary education, male (% ages 25 and older)             36
Share of seats in parliament, female (% held by women)                                    24
Share of seats in parliament, male (% held by men)                    

In [16]:
merged_df.head()

,year,Rank,country,Life evaluation (3-year average),Access to clean fuels and technologies for cooking (% of population) [EG.CFT.ACCS.ZS],"Access to clean fuels and technologies for cooking, rural (% of rural population) [EG.CFT.ACCS.RU.ZS]","Access to clean fuels and technologies for cooking, urban (% of urban population) [EG.CFT.ACCS.UR.ZS]",Access to electricity (% of population) [EG.ELC.ACCS.ZS],"Access to electricity, rural (% of rural population) [EG.ELC.ACCS.RU.ZS]","Access to electricity, urban (% of urban population) [EG.ELC.ACCS.UR.ZS]",...,"Gross National Income Per Capita, male (2021 PPP$)","Labour force participation rate, female (% ages 15 and older)","Labour force participation rate, male (% ages 15 and older)",Life Expectancy at Birth (years),"Life Expectancy at Birth, female (years)","Life Expectancy at Birth, male (years)","Population with at least some secondary education, female (% ages 25 and older)","Population with at least some secondary education, male (% ages 25 and older)","Share of seats in parliament, female (% held by women)","Share of seats in parliament, male (% held by men)"
1,2023,143.0,Afghanistan,1.721,38.6,20.4,87.0,85.3,81.4,96.0,...,3198.205,24.48,88.62,66.035,67.536,64.467,7.042,24.077,27.215,72.785
2,2022,137.0,Afghanistan,1.859,37.1,19.2,86.3,85.3,81.7,95.9,...,3205.387,23.45,88.62,65.617,67.236,63.941,7.042,24.077,27.215,72.785
3,2021,146.0,Afghanistan,2.404,35.9,18.0,85.4,97.7,97.1,99.5,...,3412.198,23.30,76.96,60.417,64.079,57.052,6.400,14.870,27.215,72.785
4,2020,150.0,Afghanistan,2.523,34.6,16.8,84.9,97.7,97.1,99.5,...,4615.328,16.50,66.84,61.454,64.656,58.455,7.340,26.026,27.215,72.785
5,2019,153.0,Afghanistan,2.567,33.3,15.7,83.9,97.7,97.1,99.5,...,4859.986,18.05,68.95,62.941,66.144,59.910,6.855,25.990,27.244,72.756


#### Any rows that don't contain this column are country/year pairs that don't have data available so they are excluded ####

In [17]:
threshold = 0.30
merged_df = merged_df.loc[:, merged_df.isnull().mean() <= threshold]
merged_df = merged_df[
    merged_df['Access to electricity, rural (% of rural population) [EG.ELC.ACCS.RU.ZS]'].notna()
]

In [18]:
merged_df.isnull().sum()

year                                                                                      0
Rank                                                                                      0
country                                                                                   0
Life evaluation (3-year average)                                                          0
Access to clean fuels and technologies for cooking (% of population) [EG.CFT.ACCS.ZS]    26
                                                                                         ..
Life Expectancy at Birth, male (years)                                                    0
Population with at least some secondary education, female (% ages 25 and older)          13
Population with at least some secondary education, male (% ages 25 and older)            13
Share of seats in parliament, female (% held by women)                                    0
Share of seats in parliament, male (% held by men)                              

In [19]:
rows_with_nulls = merged_df[merged_df.isnull().any(axis=1)]
rows_with_nulls

,year,Rank,country,Life evaluation (3-year average),Access to clean fuels and technologies for cooking (% of population) [EG.CFT.ACCS.ZS],"Access to clean fuels and technologies for cooking, rural (% of rural population) [EG.CFT.ACCS.RU.ZS]","Access to clean fuels and technologies for cooking, urban (% of urban population) [EG.CFT.ACCS.UR.ZS]",Access to electricity (% of population) [EG.ELC.ACCS.ZS],"Access to electricity, rural (% of rural population) [EG.ELC.ACCS.RU.ZS]","Access to electricity, urban (% of urban population) [EG.ELC.ACCS.UR.ZS]",...,"Gross National Income Per Capita, male (2021 PPP$)","Labour force participation rate, female (% ages 15 and older)","Labour force participation rate, male (% ages 15 and older)",Life Expectancy at Birth (years),"Life Expectancy at Birth, female (years)","Life Expectancy at Birth, male (years)","Population with at least some secondary education, female (% ages 25 and older)","Population with at least some secondary education, male (% ages 25 and older)","Share of seats in parliament, female (% held by women)","Share of seats in parliament, male (% held by men)"
4,2020,150.0,Afghanistan,2.523,34.60,16.8,84.90,97.7,97.1,99.5,...,4615.328,16.50,66.84,61.454,64.656,58.455,7.340,26.026,27.215,72.785
5,2019,153.0,Afghanistan,2.567,33.30,15.7,83.90,97.7,97.1,99.5,...,4859.986,18.05,68.95,62.941,66.144,59.910,6.855,25.990,27.244,72.756
6,2018,154.0,Afghanistan,3.203,31.70,14.5,82.95,93.4,91.6,98.8,...,4471.253,19.61,71.06,62.443,65.942,59.175,6.369,25.954,27.445,72.555
7,2017,145.0,Afghanistan,3.632,30.15,13.4,81.90,97.7,97.1,99.5,...,4481.192,21.17,73.16,62.406,65.606,59.388,5.884,25.917,27.445,72.555
8,2016,141.0,Afghanistan,3.794,28.90,12.3,80.60,97.7,97.1,99.5,...,4480.793,22.71,74.47,62.646,64.640,60.661,5.230,23.630,27.445,72.555
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1959,2021,144.0,Zimbabwe,2.995,30.30,7.1,77.80,49.0,31.6,85.3,...,3344.835,60.01,71.59,60.135,62.289,57.717,87.430,92.319,34.571,65.429
1960,2020,149.0,Zimbabwe,3.145,30.30,6.8,78.10,52.7,37.1,85.7,...,3519.091,60.35,71.73,61.530,63.747,59.091,86.325,92.241,34.571,65.429
1961,2019,151.0,Zimbabwe,3.299,30.10,6.5,78.40,46.7,28.3,85.4,...,3914.475,60.69,71.88,61.060,63.049,58.830,85.219,92.162,34.571,65.429
1967,2012,103.0,Zimbabwe,4.827,30.00,5.1,80.70,44.0,23.8,85.4,...,4058.570,58.10,76.76,55.386,56.609,54.066,55.280,66.820,17.891,82.109


#### This block will fill data with the closest available year of data for a country, if no data is available, it falls back to the global mean of the column ####

In [20]:
key_cols = ["country", "year"]

# 1) Convert all non-key columns to numeric
for col in merged_df.columns:
    if col not in key_cols:
        merged_df[col] = pd.to_numeric(merged_df[col], errors="coerce")

# 2) Sort so year order is consistent inside each country
merged_df = merged_df.sort_values(["country", "year"]).reset_index(drop=True)

value_cols = merged_df.columns.difference(key_cols)
global_means = merged_df[value_cols].mean()

def fill_nearest_year(group: pd.DataFrame, cols) -> pd.DataFrame:
    years = group["year"].to_numpy()
    out = group.copy()

    for col in cols:
        s = out[col].to_numpy()
        na_mask = np.isnan(s)
        if not na_mask.any():
            continue

        ok_mask = ~na_mask
        if not ok_mask.any():
            #Fallback to global mean
            gm = global_means[col]
            if not np.isnan(gm):
                out.loc[:, col] = gm
            continue

        ok_years = years[ok_mask]
        ok_vals  = s[ok_mask]

        # For each missing year, find insertion point among existing years
        idx_missing = np.flatnonzero(na_mask)
        miss_years = years[idx_missing]
        pos = np.searchsorted(ok_years, miss_years)

        # Candidate: previous available year
        prev_pos = pos - 1
        has_prev = prev_pos >= 0

        # Candidate: next available year
        next_pos = pos
        has_next = next_pos < len(ok_years)

        # Distances to prev/next
        prev_dist = np.where(has_prev, miss_years - ok_years[np.clip(prev_pos, 0, len(ok_years)-1)], np.inf)
        next_dist = np.where(has_next, ok_years[np.clip(next_pos, 0, len(ok_years)-1)] - miss_years, np.inf)

        # Choose next if it's closer OR tied (tie-break -> future)
        choose_next = next_dist <= prev_dist
        chosen_pos = np.where(choose_next, next_pos, prev_pos)

        # Fill
        out.loc[out.index[idx_missing], col] = ok_vals[chosen_pos]

    return out

# 3) Apply per-country for all value columns
merged_df = (
    merged_df.groupby("country", group_keys=False)
             .apply(fill_nearest_year, cols=value_cols)
)

C:\Users\Nick\AppData\Local\Temp\ipykernel_3204\3717902165.py:64: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fill_nearest_year, cols=value_cols)


In [21]:
merged_df.isnull().sum()

year                                                                                     0
Rank                                                                                     0
country                                                                                  0
Life evaluation (3-year average)                                                         0
Access to clean fuels and technologies for cooking (% of population) [EG.CFT.ACCS.ZS]    0
                                                                                        ..
Life Expectancy at Birth, male (years)                                                   0
Population with at least some secondary education, female (% ages 25 and older)          0
Population with at least some secondary education, male (% ages 25 and older)            0
Share of seats in parliament, female (% held by women)                                   0
Share of seats in parliament, male (% held by men)                                       0

#### Exporting the Cleaned/Merged Data to a csv ####

In [22]:
merged_df.to_csv('Finalized_MasterList.csv', index = False)